## 1. Setup and Prerequisite Checks

Import libraries, define constants like thresholds and persistence parameters, and check for required input files from 03b, stopping with a clear checklist if missing.

In [39]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

# Define constants
DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
NOTEBOOKS_DIR = Path('../notebooks')

# Thresholds (adapt from 03c)
RESIDUAL_MAGNITUDE_THRESHOLD = 1.5     # was 0.5 — aligns to p90 of abs_residual for tvoc
ROBUST_Z_THRESHOLD = 3.5               # was 2.0 — p90 level, much more selective
RELATIVE_RESIDUAL_THRESHOLD = 0.2      # keep value but we will DISABLE it in logic below
MIN_PERSISTENCE_HOURS = 6
PERSISTENCE_WINDOW_HOURS = 24
PERSISTENCE_RATIO_THRESHOLD = 0.25
MIN_NEIGHBOUR_COVERAGE = 0.5
ISOLATED_SENSOR_DISCOUNT = 0.7
INSTABILITY_THRESHOLDS = {'low': 0.1, 'high': 0.5}
SELF_HISTORY_DEVIATION_THRESHOLD = 0.3
SELF_HISTORY_SLOPE_THRESHOLD = 0.1
SELF_HISTORY_Z_THRESHOLD = 3.5   # normalised self z-score, same scale as ROBUST_Z_THRESHOLD
DECISION_WINDOW_HOURS = 24

# Required files from 03b
REQUIRED_03B_FILES = [
    'sensor_hour_features.csv',
    'sensor_window_features.csv',
    'feature_dictionary.csv',
    'sensor_neighbour_metadata.csv',
    'model_metric_list.csv'
]

# Optional files from 03c
OPTIONAL_03C_FILES = [
    'sensor_window_scores.csv',
    'fault_rule_parameters.json',
    'score_diagnostics.csv'
]

# Check for required files
missing_files = []
for file in REQUIRED_03B_FILES:
    if not (DATA_DIR / file).exists():
        missing_files.append(file)

if missing_files:
    print("Missing required files from 03b:")
    for file in missing_files:
        print(f"  - {file}")
    raise ValueError("Cannot proceed without required upstream files. Please run notebooks 01-03b first.")

print("All required 03b files found. Proceeding...")

# Check optional 03c files
available_03c = {}
for file in OPTIONAL_03C_FILES:
    available_03c[file] = (DATA_DIR / file).exists()
    print(f"03c file {file}: {'available' if available_03c[file] else 'not available'}")

All required 03b files found. Proceeding...
03c file sensor_window_scores.csv: available
03c file fault_rule_parameters.json: available
03c file score_diagnostics.csv: available


## 2. Load 03b and Optional 03c Files

Load and parse CSV and JSON files from 03b and 03c if available.

In [40]:
# --- 03b required files ---
sensor_hour_features = pd.read_csv(DATA_DIR / 'sensor_hour_features.csv')
sensor_window_features = pd.read_csv(DATA_DIR / 'sensor_window_features.csv')
feature_dictionary = pd.read_csv(DATA_DIR / 'feature_dictionary.csv')
sensor_neighbour_metadata = pd.read_csv(DATA_DIR / 'sensor_neighbour_metadata.csv')
model_metric_list = pd.read_csv(DATA_DIR / 'model_metric_list.csv')

print("Loaded 03b files:")
print(f"  sensor_hour_features:      {sensor_hour_features.shape}")
print(f"  sensor_window_features:    {sensor_window_features.shape}")
print(f"  feature_dictionary:        {feature_dictionary.shape}")
print(f"  sensor_neighbour_metadata: {sensor_neighbour_metadata.shape}")
print(f"  model_metric_list:         {model_metric_list.shape}")

# --- 03c optional files ---
sensor_window_scores_03c = None
fault_rule_parameters = None
score_diagnostics = None

if available_03c['sensor_window_scores.csv']:
    sensor_window_scores_03c = pd.read_csv(DATA_DIR / 'sensor_window_scores.csv')
    print(f"\n  sensor_window_scores (03c): {sensor_window_scores_03c.shape}")

if available_03c['fault_rule_parameters.json']:
    with open(DATA_DIR / 'fault_rule_parameters.json', 'r') as f:
        fault_rule_parameters = json.load(f)
    print(f"  fault_rule_parameters (03c): loaded")

if available_03c['score_diagnostics.csv']:
    score_diagnostics = pd.read_csv(DATA_DIR / 'score_diagnostics.csv')
    print(f"  score_diagnostics (03c):    {score_diagnostics.shape}")

# --- Quick peek at key column names (will use in Cell 3 for mapping) ---
print("\nsensor_hour_features columns (first 20):")
print(list(sensor_hour_features.columns[:20]))

print("\nsensor_window_features columns (first 20):")
print(list(sensor_window_features.columns[:20]))

print("\nsensor_neighbour_metadata columns:")
print(list(sensor_neighbour_metadata.columns))

print("\nmodel_metric_list columns and values:")
print(model_metric_list)

Loaded 03b files:
  sensor_hour_features:      (25344, 573)
  sensor_window_features:    (1056, 105)
  feature_dictionary:        (571, 4)
  sensor_neighbour_metadata: (33, 3)
  model_metric_list:         (14, 1)

  sensor_window_scores (03c): (1056, 23)
  fault_rule_parameters (03c): loaded
  score_diagnostics (03c):    (15, 2)

sensor_hour_features columns (first 20):
['timestamp_utc', 'ateccid', 'neighbour_mode', 'neighbour_count', 'effective_neighbour_count', 'fallback_reason', 'tvoc_raw', 'tvoc_neighbour_mean', 'tvoc_neighbour_median', 'tvoc_neighbour_std', 'tvoc_neighbour_mad', 'tvoc_neighbour_count', 'tvoc_neighbour_coverage_ratio', 'co2_raw', 'co2_neighbour_mean', 'co2_neighbour_median', 'co2_neighbour_std', 'co2_neighbour_mad', 'co2_neighbour_count', 'co2_neighbour_coverage_ratio']

sensor_window_features columns (first 20):
['ateccid', 'window_start', 'window_end', 'hours_with_data', 'mean_neighbour_coverage', 'min_neighbour_coverage', 'fallback_hours', 'tvoc_resid_max', 'tvo

## 3. Schema Validation

Validate data schemas, including timestamp parsing, one row per sensor-hour/window, 24-hour window duration, metric list, and neighbour metadata fields, with explicit column mapping if needed.

In [41]:
# -------------------------------------------------------------------
# Step 1: Lock in actual column names from what we found in Cell 2
# -------------------------------------------------------------------
# Override the initial assumptions with the real column names
TIME_COL = 'timestamp_utc'          # was assumed: 'timestamputc'
WINDOW_START_COL = 'window_start'   # was assumed: 'windowstart'
WINDOW_END_COL = 'window_end'       # was assumed: 'windowend'
SENSOR_ID_COL = 'ateccid'              # was assumed: 'ateccid'
NEIGHBOUR_COUNT_COL = 'neighbour_count'
NEIGHBOUR_MODE_COL = 'neighbour_mode'
EFFECTIVE_NEIGHBOUR_COUNT_COL = 'effective_neighbour_count'
FALLBACK_REASON_COL = 'fallback_reason'

print("=== Column Mapping (assumed → actual) ===")
print(f"  timestamputc         → {TIME_COL}")
print(f"  windowstart          → {WINDOW_START_COL}")
print(f"  windowend            → {WINDOW_END_COL}")
print(f"  neighbourcount       → {NEIGHBOUR_COUNT_COL}")

# -------------------------------------------------------------------
# Step 2: Confirm key columns exist in each dataframe
# -------------------------------------------------------------------
def assert_columns_exist(df, cols, df_name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{df_name}] Missing expected columns: {missing}")
    print(f"  [{df_name}] OK — all required columns present")

assert_columns_exist(
    sensor_hour_features,
    [SENSOR_ID_COL, TIME_COL, NEIGHBOUR_COUNT_COL],
    'sensor_hour_features'
)
assert_columns_exist(
    sensor_window_features,
    [SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL],
    'sensor_window_features'
)
assert_columns_exist(
    sensor_neighbour_metadata,
    [SENSOR_ID_COL, NEIGHBOUR_COUNT_COL],
    'sensor_neighbour_metadata'
)

# -------------------------------------------------------------------
# Step 3: Parse timestamps
# -------------------------------------------------------------------
sensor_hour_features[TIME_COL] = pd.to_datetime(sensor_hour_features[TIME_COL], utc=True)
sensor_window_features[WINDOW_START_COL] = pd.to_datetime(sensor_window_features[WINDOW_START_COL], utc=True)
sensor_window_features[WINDOW_END_COL] = pd.to_datetime(sensor_window_features[WINDOW_END_COL], utc=True)
print(f"\nTimestamps parsed.")
print(f"  sensor_hour_features time range: "
      f"{sensor_hour_features[TIME_COL].min()} → {sensor_hour_features[TIME_COL].max()}")
print(f"  sensor_window_features range:    "
      f"{sensor_window_features[WINDOW_START_COL].min()} → {sensor_window_features[WINDOW_END_COL].max()}")

# -------------------------------------------------------------------
# Step 4: Validate one row per sensor-hour
# -------------------------------------------------------------------
hourly_dupes = sensor_hour_features.groupby([SENSOR_ID_COL, TIME_COL]).size()
if hourly_dupes.max() > 1:
    print(f"\nWARNING: Multiple rows per sensor-hour found (max={hourly_dupes.max()})")
else:
    print(f"\nValidated: one row per sensor-hour  ✓")

# -------------------------------------------------------------------
# Step 5: Validate one row per sensor-window
# -------------------------------------------------------------------
window_dupes = sensor_window_features.groupby([SENSOR_ID_COL, WINDOW_START_COL]).size()
if window_dupes.max() > 1:
    print(f"WARNING: Multiple rows per sensor-window found (max={window_dupes.max()})")
else:
    print(f"Validated: one row per sensor-window ✓")

# -------------------------------------------------------------------
# Step 6: Validate 24-hour window duration
# -------------------------------------------------------------------
sensor_window_features['_duration_hours'] = (
    sensor_window_features[WINDOW_END_COL] - sensor_window_features[WINDOW_START_COL]
).dt.total_seconds() / 3600

non_24 = sensor_window_features[~np.isclose(sensor_window_features['_duration_hours'], 24, atol=0.1)]
if len(non_24) > 0:
    print(f"WARNING: {len(non_24)} windows are not exactly 24 hours")
    print(non_24[[SENSOR_ID_COL, WINDOW_START_COL, '_duration_hours']].head())
else:
    print(f"Validated: all windows are exactly 24 hours ✓")

sensor_window_features = sensor_window_features.drop(columns=['_duration_hours'])

# -------------------------------------------------------------------
# Step 7: Extract and confirm metric list
# -------------------------------------------------------------------
metrics = model_metric_list['metric'].tolist()
print(f"\nModelling metrics ({len(metrics)}): {metrics}")

# -------------------------------------------------------------------
# Step 8: Confirm hourly column naming convention for metrics
# -------------------------------------------------------------------
print("\n=== Hourly feature column patterns per metric ===")
column_patterns = {
    'raw':                     '{metric}_raw',
    'neighbour_mean':          '{metric}_neighbour_mean',
    'neighbour_median':        '{metric}_neighbour_median',
    'neighbour_std':           '{metric}_neighbour_std',
    'neighbour_mad':           '{metric}_neighbour_mad',
    'neighbour_coverage':      '{metric}_neighbour_coverage_ratio',
}
window_column_patterns = {
    'resid_max':   '{metric}_resid_max',
    'resid_mean':  '{metric}_resid_mean',
    'resid_median':'{metric}_resid_median',
    'resid_rms':   '{metric}_resid_rms',
    'z_max':       '{metric}_z_max',
    'z_mean':      '{metric}_z_mean',
    'z_median':    '{metric}_z_median',
}

# Spot-check one metric
spot = metrics[0]
for label, pattern in column_patterns.items():
    col = pattern.replace('{metric}', spot)
    status = "✓" if col in sensor_hour_features.columns else "✗ MISSING"
    print(f"  [{spot}] {label}: {col}  {status}")

print(f"\n=== Window feature column patterns (spot check: {spot}) ===")
for label, pattern in window_column_patterns.items():
    col = pattern.replace('{metric}', spot)
    status = "✓" if col in sensor_window_features.columns else "✗ MISSING"
    print(f"  [{spot}] {label}: {col}  {status}")

# -------------------------------------------------------------------
# Step 9: Neighbour metadata summary
# -------------------------------------------------------------------
print(f"\n=== Neighbour metadata summary ===")
print(sensor_neighbour_metadata[[SENSOR_ID_COL, NEIGHBOUR_COUNT_COL]].sort_values(
    NEIGHBOUR_COUNT_COL, ascending=False
).to_string(index=False))

=== Column Mapping (assumed → actual) ===
  timestamputc         → timestamp_utc
  windowstart          → window_start
  windowend            → window_end
  neighbourcount       → neighbour_count
  [sensor_hour_features] OK — all required columns present
  [sensor_window_features] OK — all required columns present
  [sensor_neighbour_metadata] OK — all required columns present

Timestamps parsed.
  sensor_hour_features time range: 2025-08-15 00:00:00+00:00 → 2025-09-15 23:00:00+00:00
  sensor_window_features range:    2025-08-15 00:00:00+00:00 → 2025-09-16 00:00:00+00:00

Validated: one row per sensor-hour  ✓
Validated: one row per sensor-window ✓
Validated: all windows are exactly 24 hours ✓

Modelling metrics (14): ['tvoc', 'co2', 'temp', 'pressure', 'lux', 'snd', 'humid', 'aqi1', 'aqi2', 'blue_relative', 'clear_relative', 'green_relative', 'red_relative', 'voc_resistance']

=== Hourly feature column patterns per metric ===
  [tvoc] raw: tvoc_raw  ✓
  [tvoc] neighbour_mean: tvoc_neig

## 4. Metric Registry

Create and export a metric registry table documenting existing features for each metric in model_metric_list.csv, including raw values, neighbour features, residuals, and self-history baselines.

In [42]:
# Documents every available feature column per metric across both hourly and window frames

# -------------------------------------------------------------------
# Define column patterns to look up per metric
# -------------------------------------------------------------------
HOURLY_PATTERNS = {
    'raw':                      '{m}_raw',
    'neighbour_mean':           '{m}_neighbour_mean',
    'neighbour_median':         '{m}_neighbour_median',
    'neighbour_std':            '{m}_neighbour_std',
    'neighbour_mad':            '{m}_neighbour_mad',
    'neighbour_count':          '{m}_neighbour_count',
    'neighbour_coverage_ratio': '{m}_neighbour_coverage_ratio',
    'robust_z':                 '{m}_robust_z',
    'residual':                 '{m}_residual',
    'self_mean':                '{m}_self_mean',
    'self_median':              '{m}_self_median',
    'self_std':                 '{m}_self_std',
    'self_mad':                 '{m}_self_mad',
}

WINDOW_PATTERNS = {
    'resid_max':    '{m}_resid_max',
    'resid_mean':   '{m}_resid_mean',
    'resid_median': '{m}_resid_median',
    'resid_rms':    '{m}_resid_rms',
    'z_max':        '{m}_z_max',
    'z_mean':       '{m}_z_mean',
    'z_median':     '{m}_z_median',
    'z_rms':        '{m}_z_rms',
}

# -------------------------------------------------------------------
# Build registry
# -------------------------------------------------------------------
registry_rows = []

for m in metrics:
    row = {'metric': m}

    # Check hourly columns
    for label, pattern in HOURLY_PATTERNS.items():
        col = pattern.replace('{m}', m)
        row[f'hourly_{label}'] = col if col in sensor_hour_features.columns else None

    # Check window columns
    for label, pattern in WINDOW_PATTERNS.items():
        col = pattern.replace('{m}', m)
        row[f'window_{label}'] = col if col in sensor_window_features.columns else None

    # Determine basis type for this metric
    # (based on whether raw and neighbour columns both exist)
    has_raw = row['hourly_raw'] is not None
    has_neighbour = row['hourly_neighbour_median'] is not None
    has_window_resid = row['window_resid_max'] is not None
    has_window_z = row['window_z_max'] is not None

    if has_neighbour:
        row['evidence_basis'] = 'neighbour_aware'
    elif has_raw:
        row['evidence_basis'] = 'self_history_only'
    else:
        row['evidence_basis'] = 'no_raw_column'

    row['has_raw'] = has_raw
    row['has_neighbour_cols'] = has_neighbour
    row['has_window_resid'] = has_window_resid
    row['has_window_z'] = has_window_z

    registry_rows.append(row)

metric_registry_df = pd.DataFrame(registry_rows)

# -------------------------------------------------------------------
# Print summary table
# -------------------------------------------------------------------
summary_cols = [
    'metric', 'evidence_basis',
    'has_raw', 'has_neighbour_cols',
    'has_window_resid', 'has_window_z',
    'hourly_raw', 'hourly_neighbour_median',
    'hourly_robust_z', 'hourly_residual',
    'window_resid_max', 'window_z_max'
]
print("=== Metric Registry Summary ===")
print(metric_registry_df[summary_cols].to_string(index=False))

# -------------------------------------------------------------------
# Flag any metrics with missing raw column (cannot model)
# -------------------------------------------------------------------
no_raw = metric_registry_df[metric_registry_df['has_raw'] == False]['metric'].tolist()
if no_raw:
    print(f"\nWARNING: {len(no_raw)} metrics have no raw column — will be skipped in modelling:")
    for m in no_raw:
        print(f"  - {m}")
else:
    print(f"\nAll {len(metrics)} metrics have a raw column ✓")

# -------------------------------------------------------------------
# Flag any metrics where neighbour columns are missing
# (these will use self-history fallback at metric level)
# -------------------------------------------------------------------
no_neigh = metric_registry_df[metric_registry_df['has_neighbour_cols'] == False]['metric'].tolist()
if no_neigh:
    print(f"\nMetrics with no neighbour columns (self-history fallback):")
    for m in no_neigh:
        print(f"  - {m}")
else:
    print(f"All {len(metrics)} metrics have neighbour columns ✓")

# -------------------------------------------------------------------
# Export
# -------------------------------------------------------------------
metric_registry_df.to_csv(DATA_DIR / 'metric_registry.csv', index=False)
print(f"\nExported metric_registry.csv ({len(metric_registry_df)} rows)")

# -------------------------------------------------------------------
# Also build a quick-lookup dict for downstream cells
# -------------------------------------------------------------------
metric_registry = {row['metric']: row for _, row in metric_registry_df.iterrows()}
print("metric_registry dict built for downstream use ✓")

=== Metric Registry Summary ===
        metric  evidence_basis  has_raw  has_neighbour_cols  has_window_resid  has_window_z         hourly_raw         hourly_neighbour_median         hourly_robust_z hourly_residual         window_resid_max         window_z_max
          tvoc neighbour_aware     True                True              True          True           tvoc_raw           tvoc_neighbour_median           tvoc_robust_z            None           tvoc_resid_max           tvoc_z_max
           co2 neighbour_aware     True                True              True          True            co2_raw            co2_neighbour_median            co2_robust_z            None            co2_resid_max            co2_z_max
          temp neighbour_aware     True                True              True          True           temp_raw           temp_neighbour_median           temp_robust_z            None           temp_resid_max           temp_z_max
      pressure neighbour_aware     True             

## 5. Metric-Specific Hourly Evidence

For each metric, build or reuse hourly evidence fields like residuals, z-scores, self-history deviations, persistence counts, and neighbour-based features, handling neighbour-availability modes.

In [43]:
# For each metric, derive per-sensor-hour evidence fields
# using neighbour columns (where available) and self-history rolling baselines.
# Output: metric_hourly_evidence dict keyed by metric name

# -------------------------------------------------------------------
# Neighbour tier classification
# -------------------------------------------------------------------
# 0 neighbours  → self_history_only
# 1–2 neighbours → low_confidence
# 3+  neighbours → neighbour_aware
def classify_basis(n):
    if n == 0:
        return 'self_history_only'
    elif n <= 2:
        return 'low_confidence'
    else:
        return 'neighbour_aware'

# -------------------------------------------------------------------
# Core function: build hourly evidence for one metric
# -------------------------------------------------------------------
def build_metric_hourly_evidence(sensor_hour_features, metric, reg):
    """
    Returns a tidy DataFrame with standardised evidence column names
    (no metric prefix inside the returned df — metric is a separate column).
    """
    raw_col      = reg['hourly_raw']
    neigh_med    = reg['hourly_neighbour_median']
    neigh_std    = f"{metric}_neighbour_std"
    neigh_mad    = f"{metric}_neighbour_mad"
    neigh_cov    = f"{metric}_neighbour_coverage_ratio"
    robust_z_col = reg['hourly_robust_z']

    # Pull only needed columns + identifiers
    keep = [SENSOR_ID_COL, TIME_COL, NEIGHBOUR_COUNT_COL]
    for c in [raw_col, neigh_med, neigh_std, neigh_mad, neigh_cov, robust_z_col]:
        if c and c in sensor_hour_features.columns:
            keep.append(c)

    df = sensor_hour_features[keep].copy().sort_values(
        [SENSOR_ID_COL, TIME_COL]
    ).reset_index(drop=True)

    # --- Basis tier ---
    df['metric']       = metric
    df['metric_basis'] = df[NEIGHBOUR_COUNT_COL].apply(classify_basis)

    # --- Raw value ---
    df['raw'] = df[raw_col] if raw_col else np.nan

    # --- Neighbour evidence ---
    if neigh_med in df.columns:
        df['neigh_median'] = df[neigh_med]
        df['abs_residual'] = (df['raw'] - df['neigh_median']).abs()
        df['relative_residual'] = df['abs_residual'] / (df['neigh_median'].abs() + 1e-6)
    else:
        df['neigh_median']       = np.nan
        df['abs_residual']       = np.nan
        df['relative_residual']  = np.nan

    df['neigh_std'] = df[neigh_std] if neigh_std in df.columns else np.nan
    df['neigh_mad'] = df[neigh_mad] if neigh_mad in df.columns else np.nan
    df['neigh_coverage'] = df[neigh_cov] if neigh_cov in df.columns else np.nan

    # Zero out neighbour evidence for sensors with no neighbours
    no_neigh_mask = df[NEIGHBOUR_COUNT_COL] == 0
    for col in ['neigh_median', 'abs_residual', 'relative_residual', 'neigh_std', 'neigh_mad']:
        df.loc[no_neigh_mask, col] = np.nan

    # --- Robust z-score (reuse existing) ---
    if robust_z_col and robust_z_col in df.columns:
        df['robust_z'] = df[robust_z_col]
    else:
        df['robust_z'] = np.nan

    # --- Self-history rolling baselines (per sensor, sorted by time) ---
    grp = df.groupby(SENSOR_ID_COL)['raw']

    # 168h (7-day) rolling median/mad as expected-behaviour baseline
    df['self_rolling_median'] = grp.transform(
        lambda x: x.rolling(168, min_periods=24).median()
    )
    df['self_rolling_mad'] = grp.transform(
        lambda x: x.rolling(168, min_periods=24).apply(
            lambda v: np.median(np.abs(v - np.median(v))), raw=True
        )
    )

    # Self-history deviation: abs(raw - self rolling median)
    df['self_deviation'] = (df['raw'] - df['self_rolling_median']).abs()

    # Self-history slope: average hourly change over last 24h
    # Using diff(24)/24 — fast and interpretable
    df['self_slope'] = grp.transform(lambda x: x.diff(24) / 24.0)

    # Self z-score: scale-invariant version of self_deviation
    df['self_z'] = df['self_deviation'] / (df['self_rolling_mad'].fillna(1e-6) + 1e-6)

    # --- Instability: rolling 24h std of abs_residual (or raw if no neighbours) ---
    residual_for_instability = df['abs_residual'].where(
        df[NEIGHBOUR_COUNT_COL] > 0, df['self_deviation']
    )
    df['rolling_instability'] = (
        df.groupby(SENSOR_ID_COL)[residual_for_instability.name]
        .transform(lambda x: x.rolling(24, min_periods=6).std())
        if residual_for_instability.name in df.columns
        else df.groupby(SENSOR_ID_COL)['raw']
        .transform(lambda x: x.rolling(24, min_periods=6).std())
    )

    # Compute instability directly to avoid name resolution issues
    instability_input = np.where(
        df[NEIGHBOUR_COUNT_COL].values > 0,
        df['abs_residual'].values,
        df['self_deviation'].values
    )
    df['_instability_input'] = instability_input
    df['rolling_instability'] = df.groupby(SENSOR_ID_COL)['_instability_input'].transform(
        lambda x: x.rolling(24, min_periods=6).std()
    )
    df = df.drop(columns=['_instability_input'])

    # --- Threshold exceedance flag ---
    # relative_residual is excluded — numerically unstable when neigh_median ≈ 0
    # abs_residual is excluded — not scale-invariant across metrics
    # robust_z is the correct scale-normalised signal for neighbour-aware sensors
    # robust_z for neighbour-aware sensors (scale-normalised)
    df['threshold_exceed'] = (
        df['robust_z'].fillna(0).abs() > ROBUST_Z_THRESHOLD
    ).astype(int)

    # For self-history-only sensors: use normalised self z-score
    df.loc[no_neigh_mask, 'threshold_exceed'] = (
        df.loc[no_neigh_mask, 'self_z'].fillna(0).abs() > SELF_HISTORY_Z_THRESHOLD
    ).astype(int)

    # --- Persistence count: rolling 24h sum of threshold exceedance per sensor ---
    df['persistence_count'] = df.groupby(SENSOR_ID_COL)['threshold_exceed'].transform(
        lambda x: x.rolling(PERSISTENCE_WINDOW_HOURS, min_periods=1).sum()
    )

    # --- Drop intermediate source columns, keep clean evidence schema ---
    drop_cols = [c for c in [raw_col, neigh_med, neigh_std, neigh_mad, neigh_cov, robust_z_col]
                 if c and c in df.columns]
    df = df.drop(columns=drop_cols)

    return df

# -------------------------------------------------------------------
# Build evidence for all metrics
# -------------------------------------------------------------------
metric_hourly_evidence = {}

for metric in metrics:
    reg = metric_registry[metric]
    print(f"Building hourly evidence for {metric}...")
    metric_hourly_evidence[metric] = build_metric_hourly_evidence(
        sensor_hour_features, metric, reg
    )

print(f"\nDone. Evidence built for {len(metric_hourly_evidence)} metrics.")

# -------------------------------------------------------------------
# Spot check: one metric, one sensor
# -------------------------------------------------------------------
spot_metric = metrics[0]
spot_df = metric_hourly_evidence[spot_metric]

print(f"\nSpot check — {spot_metric}:")
print(f"  Shape: {spot_df.shape}")
print(f"  Columns: {list(spot_df.columns)}")
print(f"  metric_basis value counts:")
print(spot_df['metric_basis'].value_counts().to_string())
print(f"\n  threshold_exceed rate: {spot_df['threshold_exceed'].mean():.3f}")
print(f"  persistence_count range: {spot_df['persistence_count'].min():.0f} – "
      f"{spot_df['persistence_count'].max():.0f}")
print(f"\n  Sample rows:")
print(spot_df[[SENSOR_ID_COL, TIME_COL, 'metric_basis', 'raw',
               'abs_residual', 'robust_z', 'self_deviation',
               'rolling_instability', 'threshold_exceed',
               'persistence_count']].head(8).to_string(index=False))

Building hourly evidence for tvoc...
Building hourly evidence for co2...
Building hourly evidence for temp...
Building hourly evidence for pressure...
Building hourly evidence for lux...
Building hourly evidence for snd...
Building hourly evidence for humid...
Building hourly evidence for aqi1...
Building hourly evidence for aqi2...
Building hourly evidence for blue_relative...
Building hourly evidence for clear_relative...
Building hourly evidence for green_relative...
Building hourly evidence for red_relative...
Building hourly evidence for voc_resistance...

Done. Evidence built for 14 metrics.

Spot check — tvoc:
  Shape: (25344, 21)
  Columns: ['ateccid', 'timestamp_utc', 'neighbour_count', 'metric', 'metric_basis', 'raw', 'neigh_median', 'abs_residual', 'relative_residual', 'neigh_std', 'neigh_mad', 'neigh_coverage', 'robust_z', 'self_rolling_median', 'self_rolling_mad', 'self_deviation', 'self_slope', 'self_z', 'rolling_instability', 'threshold_exceed', 'persistence_count']
  me

threshold_exceed rate should be low (0.05–0.15 is healthy), however,  after running the above cell, it is 0.449.

this is too high and will lead to inflated fault rates downstream. Before we build Cell 6, we need a quick diagnostic to understand which threshold condition is the main driver, then we can recalibrate before aggregating to windows. 

In [44]:
# Identify which condition(s) are firing most before we proceed

spot_df = metric_hourly_evidence['tvoc']

# Check individual component rates
neigh_rows = spot_df[spot_df['metric_basis'] != 'self_history_only']
self_rows  = spot_df[spot_df['metric_basis'] == 'self_history_only']

print("=== Threshold exceedance component analysis per metric_basis ===")
for m in metrics:
    df_m = metric_hourly_evidence[m]
    neigh = df_m[df_m['metric_basis'] != 'self_history_only']
    self_ = df_m[df_m['metric_basis'] == 'self_history_only']
    neigh_rate = (neigh['robust_z'].fillna(0).abs() > ROBUST_Z_THRESHOLD).mean()
    self_rate  = (self_['self_z'].fillna(0).abs() > SELF_HISTORY_Z_THRESHOLD).mean()
    overall    = df_m['threshold_exceed'].mean()
    print(f"  {m:<20}  neigh_z_rate={neigh_rate:.3f}  self_z_rate={self_rate:.3f}  stored={overall:.3f}")

=== Threshold exceedance component analysis per metric_basis ===
  tvoc                  neigh_z_rate=0.075  self_z_rate=0.231  stored=0.122
  co2                   neigh_z_rate=0.120  self_z_rate=0.183  stored=0.139
  temp                  neigh_z_rate=0.192  self_z_rate=0.140  stored=0.176
  pressure              neigh_z_rate=0.082  self_z_rate=0.221  stored=0.124
  lux                   neigh_z_rate=0.174  self_z_rate=0.208  stored=0.184
  snd                   neigh_z_rate=0.180  self_z_rate=0.261  stored=0.204
  humid                 neigh_z_rate=0.188  self_z_rate=0.150  stored=0.177
  aqi1                  neigh_z_rate=0.157  self_z_rate=0.085  stored=0.135
  aqi2                  neigh_z_rate=0.139  self_z_rate=0.068  stored=0.118
  blue_relative         neigh_z_rate=0.162  self_z_rate=0.201  stored=0.174
  clear_relative        neigh_z_rate=0.158  self_z_rate=0.229  stored=0.180
  green_relative        neigh_z_rate=0.169  self_z_rate=0.216  stored=0.183
  red_relative         

In [ ]:
print("=== Currently active threshold constants ===")
print(f"  RESIDUAL_MAGNITUDE_THRESHOLD : {RESIDUAL_MAGNITUDE_THRESHOLD}")
print(f"  ROBUST_Z_THRESHOLD           : {ROBUST_Z_THRESHOLD}")
print(f"  RELATIVE_RESIDUAL_THRESHOLD  : {RELATIVE_RESIDUAL_THRESHOLD}")
print()

# Re-check tvoc with explicit values (bypasses any constant confusion)
spot_df = metric_hourly_evidence['tvoc']
neigh_rows = spot_df[spot_df['metric_basis'] != 'self_history_only']

cond_abs = neigh_rows['abs_residual'].fillna(0) > 1.5
cond_z   = neigh_rows['robust_z'].fillna(0).abs() > 3.5

print(f"  abs_residual > 1.5 (hardcoded): {cond_abs.mean():.3f}  ({cond_abs.sum()} rows)")
print(f"  robust_z     > 3.5 (hardcoded): {cond_z.mean():.3f}  ({cond_z.sum()} rows)")
print(f"  ANY of above:                   {(cond_abs | cond_z).mean():.3f}")
print()

# Check what the stored threshold_exceed actually used
print(f"  Stored threshold_exceed rate (all rows): {spot_df['threshold_exceed'].mean():.3f}")
print(f"  Stored threshold_exceed rate (neigh only): {neigh_rows['threshold_exceed'].mean():.3f}")

## 6. Metric-Specific Window Evidence

Aggregate hourly evidence into sensor × 24-hour window × metric summaries, including max residuals, persistence ratios, and coverage metrics, then export to metric_window_evidence.csv.

In [45]:
# Aggregate metric hourly evidence into sensor × 24-hour window × metric summaries

def build_metric_window_evidence(hourly_df, metric, sensor_window_features):
    """
    Aggregates hourly evidence for one metric into per-sensor-window rows.
    Joins cleanly onto window boundaries without fan-out.
    """
    # Ensure timestamps are tz-aware
    hourly_df = hourly_df.copy()
    hourly_df[TIME_COL] = pd.to_datetime(hourly_df[TIME_COL], utc=True)

    win = sensor_window_features[[SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL]].copy()
    win[WINDOW_START_COL] = pd.to_datetime(win[WINDOW_START_COL], utc=True)
    win[WINDOW_END_COL]   = pd.to_datetime(win[WINDOW_END_COL], utc=True)

    # Assign each hourly row to its window via merge + filter
    merged = hourly_df.merge(win, on=SENSOR_ID_COL, how='left')
    merged = merged[
        (merged[TIME_COL] >= merged[WINDOW_START_COL]) &
        (merged[TIME_COL] <  merged[WINDOW_END_COL])
    ]

    grouped = merged.groupby([SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL])

    agg = grouped.agg(
        metric_basis           = ('metric_basis',       lambda x: x.mode().iloc[0] if len(x) > 0 else 'unknown'),
        neighbour_count        = (NEIGHBOUR_COUNT_COL,  'first'),
        mean_neigh_coverage    = ('neigh_coverage',     'mean'),
        max_abs_residual       = ('abs_residual',       'max'),
        mean_abs_residual      = ('abs_residual',       'mean'),
        max_robust_z           = ('robust_z',           lambda x: x.abs().max()),
        mean_robust_z          = ('robust_z',           lambda x: x.abs().mean()),
        max_self_deviation     = ('self_deviation',     'max'),
        max_self_z             = ('self_z',             lambda x: x.abs().max()),
        max_self_slope         = ('self_slope',         lambda x: x.abs().max()),
        rolling_instability    = ('rolling_instability','max'),
        persistence_hours      = ('persistence_count',  'max'),
        hours_with_data        = ('raw',                'count'),
        hours_scored           = ('threshold_exceed',   'sum'),
    ).reset_index()

    # Persistence ratio: fraction of hours in window where threshold was exceeded
    agg['persistence_ratio'] = agg['hours_scored'] / DECISION_WINDOW_HOURS

    # Mean neighbour coverage: fill NaN for self-history sensors
    agg['mean_neigh_coverage'] = agg['mean_neigh_coverage'].fillna(0.0)

    # Add metric label
    agg['metric'] = metric

    # Reorder columns
    col_order = [
        SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL,
        'metric', 'metric_basis', 'neighbour_count', 'mean_neigh_coverage',
        'max_abs_residual', 'mean_abs_residual',
        'max_robust_z', 'mean_robust_z',
        'max_self_deviation', 'max_self_z', 'max_self_slope',
        'rolling_instability',
        'persistence_hours', 'persistence_ratio',
        'hours_with_data', 'hours_scored',
    ]
    return agg[col_order]


# -------------------------------------------------------------------
# Build window evidence for all metrics
# -------------------------------------------------------------------
metric_window_evidence_list = []

for metric in metrics:
    print(f"Aggregating window evidence for {metric}...")
    evidence = build_metric_window_evidence(
        metric_hourly_evidence[metric],
        metric,
        sensor_window_features
    )
    metric_window_evidence_list.append(evidence)

metric_window_evidence = pd.concat(metric_window_evidence_list, ignore_index=True)

# -------------------------------------------------------------------
# Export
# -------------------------------------------------------------------
metric_window_evidence.to_csv(DATA_DIR / 'metric_window_evidence.csv', index=False)
print(f"\nExported metric_window_evidence.csv — {metric_window_evidence.shape}")

# -------------------------------------------------------------------
# Spot check
# -------------------------------------------------------------------
print(f"\n=== Spot check ===")
print(f"  Expected rows: {len(metrics)} metrics × {len(sensor_window_features)} windows = "
      f"{len(metrics) * len(sensor_window_features)}")
print(f"  Actual rows:   {len(metric_window_evidence)}")

print(f"\n  metric_basis distribution:")
print(metric_window_evidence['metric_basis'].value_counts().to_string())

print(f"\n  persistence_ratio stats:")
print(metric_window_evidence['persistence_ratio'].describe().round(3).to_string())

print(f"\n  mean_neigh_coverage stats:")
print(metric_window_evidence['mean_neigh_coverage'].describe().round(3).to_string())

print(f"\n  Sample rows (tvoc):")
sample = metric_window_evidence[metric_window_evidence['metric'] == 'tvoc'].head(4)
print(sample[[
    SENSOR_ID_COL, WINDOW_START_COL, 'metric', 'metric_basis',
    'neighbour_count', 'max_robust_z', 'persistence_hours',
    'persistence_ratio', 'hours_with_data', 'hours_scored'
]].to_string(index=False))

Aggregating window evidence for tvoc...
Aggregating window evidence for co2...
Aggregating window evidence for temp...
Aggregating window evidence for pressure...
Aggregating window evidence for lux...
Aggregating window evidence for snd...
Aggregating window evidence for humid...
Aggregating window evidence for aqi1...
Aggregating window evidence for aqi2...
Aggregating window evidence for blue_relative...
Aggregating window evidence for clear_relative...
Aggregating window evidence for green_relative...
Aggregating window evidence for red_relative...
Aggregating window evidence for voc_resistance...

Exported metric_window_evidence.csv — (14784, 19)

=== Spot check ===
  Expected rows: 14 metrics × 1056 windows = 14784
  Actual rows:   14784

  metric_basis distribution:
metric_basis
neighbour_aware      8960
self_history_only    4480
low_confidence       1344

  persistence_ratio stats:
count    14784.000
mean         0.158
std          0.222
min          0.000
25%          0.000
50

## 7. Metric-Level Rule Scores

Compute rule-based scores per metric-window using named thresholds for residuals, persistence, and coverage, generating metric_rulescore, metric_confidence, metric_binaryfaultflag, and metric_driftscore on a 0-1 scale.

In [ ]:
# Compute rule-based scores per metric-window
# Produces: metric_rule_score, metric_confidence, metric_drift_score, metric_binary_fault_flag
# All scores on 0–1 scale. Anomaly strength and confidence kept separate.

LOW_INSTABILITY_THRESHOLD = 0.10
HIGH_INSTABILITY_THRESHOLD = 0.50

def compute_metric_rule_scores(evidence_df):
    df = evidence_df.copy()

    # ------------------------------------------------------------------
    # 1. Anomaly strength components (0–1 each)
    # ------------------------------------------------------------------

    # Z-score component: scaled so ROBUST_Z_THRESHOLD → 0, 3× threshold → 1
    df['_z_score_component'] = np.clip(
        (df['max_robust_z'].fillna(0) - ROBUST_Z_THRESHOLD) /
        (ROBUST_Z_THRESHOLD * 2),
        0, 1
    )

    # Self-z component for self-history sensors
    df['_self_z_component'] = np.clip(
        (df['max_self_z'].fillna(0) - SELF_HISTORY_Z_THRESHOLD) /
        (SELF_HISTORY_Z_THRESHOLD * 2),
        0, 1
    )

    # Use z component appropriate for the basis type
    is_self = df['metric_basis'] == 'self_history_only'
    df['_anomaly_z'] = np.where(is_self, df['_self_z_component'], df['_z_score_component'])

    # Persistence component: fraction of window hours that exceeded threshold
    # Was: df['persistence_ratio'] / PERSISTENCE_RATIO_THRESHOLD  (fires at 25%)
    # Now: requires 50% of window hours to be anomalous before maxing out
    df['_persistence_component'] = np.clip(
        df['persistence_ratio'] / 0.5,
        0, 1
    )

    # Instability component
    df['_instability_component'] = np.clip(
        (df['rolling_instability'].fillna(0) - LOW_INSTABILITY_THRESHOLD) /
        (HIGH_INSTABILITY_THRESHOLD - LOW_INSTABILITY_THRESHOLD),
        0, 1
    )

    # Self-slope component (trend evidence)
    df['_slope_component'] = np.clip(
        df['max_self_slope'].fillna(0).abs() / (SELF_HISTORY_SLOPE_THRESHOLD * 3),
        0, 1
    )

    # ------------------------------------------------------------------
    # 2. Composite metric rule score (weighted, 0–1)
    # Weights: anomaly z is primary, persistence is secondary
    # ------------------------------------------------------------------
    df['metric_rule_score'] = (
        0.45 * df['_anomaly_z'] +
        0.35 * df['_persistence_component'] +
        0.10 * df['_instability_component'] +
        0.10 * df['_slope_component']
    ).clip(0, 1)

    # ------------------------------------------------------------------
    # 3. Confidence (separate from anomaly strength)
    # Based on: data coverage, neighbour count, hours with data
    # ------------------------------------------------------------------
    # Coverage confidence: how well the sensor was covered by neighbours
    df['_coverage_conf'] = np.clip(df['mean_neigh_coverage'], 0, 1)

    # Neighbour count confidence: more neighbours = more reliable reference
    df['_neigh_conf'] = np.where(
        df['metric_basis'] == 'self_history_only',
        ISOLATED_SENSOR_DISCOUNT,
        np.clip(df['neighbour_count'] / 10.0, 0.5, 1.0)
    )

    # Low-confidence tier for sensors with 1–2 neighbours
    # Was: clip(count/5, 0.4, 0.8)
    # Now: cap at 0.55 — penalises sparse neighbour reference more strongly
    df['_neigh_conf'] = np.where(
        df['metric_basis'] == 'low_confidence',
        np.clip(df['neighbour_count'] / 5.0, 0.30, 0.55),
        df['_neigh_conf']
    )

    # Data completeness confidence
    df['_data_conf'] = np.clip(df['hours_with_data'] / DECISION_WINDOW_HOURS, 0, 1)

    # Combined confidence
    df['metric_confidence'] = (
        0.40 * df['_coverage_conf'] +
        0.40 * df['_neigh_conf'] +
        0.20 * df['_data_conf']
    ).clip(0, 1)

    # ------------------------------------------------------------------
    # 4. Drift score: anomaly strength only (not multiplied by confidence)
    # Confidence gates the fault flag but does not suppress the score
    # ------------------------------------------------------------------
    df['metric_drift_score'] = df['metric_rule_score'].copy()

    # ------------------------------------------------------------------
    # 5. Binary fault flag: gated by confidence AND persistence
    # ------------------------------------------------------------------
    # Basis-specific confidence thresholds
    # low_confidence: high score required — reference is unreliable with 1-2 neighbours
    # self_history_only: lower confidence gate — structurally capped at 0.48
    # neighbour_aware: standard thresholds

    score_threshold = np.where(
        df['metric_basis'] == 'low_confidence', 0.85,
        np.where(df['metric_basis'] == 'self_history_only', 0.60, 0.55)
    )

    conf_threshold = np.where(
        df['metric_basis'] == 'self_history_only', 0.45, 0.55
    )

    df['metric_binary_fault_flag'] = (
        (df['metric_drift_score'] > score_threshold) &
        (df['metric_confidence'] >= conf_threshold) &
        (df['persistence_hours'] >= MIN_PERSISTENCE_HOURS)
    ).astype(int)

    # ------------------------------------------------------------------
    # Drop internal working columns
    # ------------------------------------------------------------------
    drop_cols = [c for c in df.columns if c.startswith('_')]
    df = df.drop(columns=drop_cols)

    return df


# ------------------------------------------------------------------
# Apply to all metric window evidence
# ------------------------------------------------------------------
metric_window_scores = compute_metric_rule_scores(metric_window_evidence)

print("=== Metric rule scores computed ===")
print(f"  Shape: {metric_window_scores.shape}")

# ------------------------------------------------------------------
# Spot check: score distributions
# ------------------------------------------------------------------
print(f"\n  metric_rule_score stats:")
print(metric_window_scores['metric_rule_score'].describe().round(3).to_string())

print(f"\n  metric_confidence stats:")
print(metric_window_scores['metric_confidence'].describe().round(3).to_string())

print(f"\n  metric_binary_fault_flag rate overall: "
      f"{metric_window_scores['metric_binary_fault_flag'].mean():.3f}")

print(f"\n  Fault flag rate per metric:")
fault_rates = (
    metric_window_scores.groupby('metric')['metric_binary_fault_flag']
    .mean()
    .sort_values(ascending=False)
    .round(3)
)
print(fault_rates.to_string())

print(f"\n  Fault flag rate per metric_basis:")
print(
    metric_window_scores.groupby('metric_basis')['metric_binary_fault_flag']
    .mean().round(3).to_string()
)

print(f"\n  Sample scored rows (tvoc):")
sample = metric_window_scores[metric_window_scores['metric'] == 'tvoc'].head(4)
print(sample[[
    SENSOR_ID_COL, WINDOW_START_COL, 'metric_basis',
    'metric_rule_score', 'metric_confidence',
    'metric_drift_score', 'metric_binary_fault_flag',
    'persistence_hours', 'persistence_ratio'
]].to_string(index=False))

=== Metric rule scores computed ===
  Shape: (14784, 23)

  metric_rule_score stats:
count    14784.000
mean         0.437
std          0.305
min          0.000
25%          0.200
50%          0.302
75%          0.712
max          1.000

  metric_confidence stats:
count    14784.000
mean         0.808
std          0.228
min          0.280
25%          0.480
50%          0.960
75%          1.000
max          1.000

  metric_binary_fault_flag rate overall: 0.254

  Fault flag rate per metric:
metric
lux               0.364
green_relative    0.359
snd               0.346
clear_relative    0.340
blue_relative     0.327
red_relative      0.291
humid             0.241
temp              0.215
aqi1              0.207
co2               0.206
aqi2              0.191
tvoc              0.180
voc_resistance    0.163
pressure          0.127

  Fault flag rate per metric_basis:
metric_basis
low_confidence       0.319
neighbour_aware      0.196
self_history_only    0.350

  Sample scored rows (tvoc):


## 8. Metric-Level Expected-Behaviour Models

Train interpretable models (e.g., Ridge, Random Forest) per metric using time-based splits, select the best model, compute validation metrics, and generate metric_model_residual_score, exporting to metric_model_summary.csv.

In [ ]:
# For each metric, train on earlier windows, validate on later windows
# Produce metric_model_residual_score per sensor-window row

# ------------------------------------------------------------------
# Feature columns available at window level per metric
# ------------------------------------------------------------------
def get_window_feature_cols(metric, sensor_window_features, metric_window_scores):
    """
    Returns two lists:
    - window_feat_cols: columns to pull from sensor_window_features via merge
    - existing_cols:    columns already present in metric_window_scores
    """
    # Metric-specific columns from sensor_window_features
    swf_candidates = [
        f"{metric}_resid_max",
        f"{metric}_resid_mean",
        f"{metric}_resid_median",
        f"{metric}_resid_rms",
        f"{metric}_z_max",
        f"{metric}_z_mean",
        f"{metric}_z_median",
    ]
    swf_shared = ['mean_neighbour_coverage', 'fallback_hours']
    window_feat_cols = [
        c for c in swf_candidates + swf_shared
        if c in sensor_window_features.columns
    ]

    # Columns already on metric_window_scores (no merge needed)
    mws_candidates = ['hours_with_data', 'mean_neigh_coverage',
                      'persistence_ratio', 'rolling_instability']
    existing_cols = [
        c for c in mws_candidates
        if c in metric_window_scores.columns
    ]

    return window_feat_cols, existing_cols

# ------------------------------------------------------------------
# Time-based train/val split
# 80% earliest windows for train, 20% latest for validation
# ------------------------------------------------------------------
def time_split(df, time_col, train_frac=0.80):
    df = df.sort_values(time_col).reset_index(drop=True)
    split_idx = int(len(df) * train_frac)
    return df.iloc[:split_idx], df.iloc[split_idx:]


# ------------------------------------------------------------------
# Train one metric model
# ------------------------------------------------------------------
def train_metric_model(metric, metric_window_scores, sensor_window_features):
    # Merge window-level features onto scored rows for this metric
    metric_rows = metric_window_scores[
        metric_window_scores['metric'] == metric
    ].copy()

    window_feat_cols, existing_cols = get_window_feature_cols(
        metric, sensor_window_features, metric_window_scores
    )
    feature_cols = window_feat_cols + existing_cols

    if not feature_cols:
        return {
            'metric': metric, 'selected_model': 'no_features',
            'train_rows': 0, 'validation_rows': 0,
            'validation_r2': np.nan, 'validation_mse': np.nan,
            'num_features': 0
        }, None

    # Merge in sensor_window_features columns only (existing_cols already present)
    if window_feat_cols:
        model_df = metric_rows.merge(
            sensor_window_features[[SENSOR_ID_COL, WINDOW_START_COL] + window_feat_cols],
            on=[SENSOR_ID_COL, WINDOW_START_COL],
            how='left'
        ).dropna(subset=feature_cols + ['metric_drift_score'])
    else:
        model_df = metric_rows.dropna(
            subset=existing_cols + ['metric_drift_score']
        ).copy()

    # Time-based split
    train_df, val_df = time_split(model_df, WINDOW_START_COL)

    if len(val_df) < 10:
        return {
            'metric': metric, 'selected_model': 'insufficient_validation',
            'train_rows': len(train_df), 'validation_rows': len(val_df),
            'validation_r2': np.nan, 'validation_mse': np.nan,
            'num_features': len(feature_cols)
        }, None

    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df['metric_drift_score']
    X_val   = val_df[feature_cols].fillna(0)
    y_val   = val_df['metric_drift_score']

    # Model selection loop
    candidates = {
        'ridge':    Ridge(alpha=1.0),
        'elasticnet': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=1000),
        'rf':       RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42),
        'gb':       GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=42),
    }

    best_name, best_model, best_r2 = None, None, -np.inf

    for name, model in candidates.items():
        try:
            model.fit(X_train, y_train)
            pred = model.predict(X_val)
            r2 = r2_score(y_val, pred)
            if r2 > best_r2:
                best_r2, best_model, best_name = r2, model, name
        except Exception:
            continue

    if best_model is None:
        return {
            'metric': metric, 'selected_model': 'all_failed',
            'train_rows': len(train_df), 'validation_rows': len(val_df),
            'validation_r2': np.nan, 'validation_mse': np.nan,
            'num_features': len(feature_cols)
        }, None

    # Final validation metrics
    val_pred = best_model.predict(X_val)
    val_mse  = mean_squared_error(y_val, val_pred)

    summary = {
        'metric':           metric,
        'selected_model':   best_name,
        'train_rows':       len(train_df),
        'validation_rows':  len(val_df),
        'validation_r2':    round(best_r2, 4),
        'validation_mse':   round(val_mse, 4),
        'num_features':     len(feature_cols),
    }

    # Generate model residual scores for ALL rows (train + val)
    X_all  = model_df[feature_cols].fillna(0)
    y_all  = model_df['metric_drift_score']
    y_pred = best_model.predict(X_all)

    # Residual score: how much the actual drift exceeds model expectation
    # Clipped 0–1, so a sensor behaving worse than expected scores high
    raw_residual = y_all.values - y_pred
    model_df = model_df.copy()
    model_df['metric_model_residual_score'] = np.clip(
        (raw_residual - 0) / (raw_residual.std() + 1e-6) * 0.25 + 0.5,
        0, 1
    )

    return summary, model_df[[SENSOR_ID_COL, WINDOW_START_COL, 'metric',
                               'metric_model_residual_score']]


# ------------------------------------------------------------------
# Run for all metrics
# ------------------------------------------------------------------
model_summaries = []
model_score_parts = []

for metric in metrics:
    print(f"Training model for {metric}...")
    summary, scored_df = train_metric_model(
        metric, metric_window_scores, sensor_window_features
    )
    model_summaries.append(summary)
    if scored_df is not None:
        model_score_parts.append(scored_df)

# ------------------------------------------------------------------
# Export model summary
# ------------------------------------------------------------------
metric_model_summary_df = pd.DataFrame(model_summaries)
metric_model_summary_df.to_csv(DATA_DIR / 'metric_model_summary.csv', index=False)

print(f"\nExported metric_model_summary.csv")
print(metric_model_summary_df[[
    'metric', 'selected_model', 'train_rows',
    'validation_rows', 'validation_r2', 'validation_mse', 'num_features'
]].to_string(index=False))

# ------------------------------------------------------------------
# Merge model residual scores back onto metric_window_scores
# ------------------------------------------------------------------
if model_score_parts:
    model_scores_combined = pd.concat(model_score_parts, ignore_index=True)

    metric_window_scores = metric_window_scores.merge(
        model_scores_combined,
        on=[SENSOR_ID_COL, WINDOW_START_COL, 'metric'],
        how='left'
    )
else:
    metric_window_scores['metric_model_residual_score'] = np.nan

# Fill any gaps with neutral 0.5
metric_window_scores['metric_model_residual_score'] = (
    metric_window_scores['metric_model_residual_score'].fillna(0.5)
)

print(f"\nmetric_model_residual_score stats:")
print(metric_window_scores['metric_model_residual_score'].describe().round(3).to_string())

Training model for tvoc...
Training model for co2...
Training model for temp...
Training model for pressure...
Training model for lux...
Training model for snd...
Training model for humid...
Training model for aqi1...
Training model for aqi2...
Training model for blue_relative...
Training model for clear_relative...
Training model for green_relative...
Training model for red_relative...
Training model for voc_resistance...

Exported metric_model_summary.csv
        metric selected_model  train_rows  validation_rows  validation_r2  validation_mse  num_features
          tvoc             gb         557              140         0.9939          0.0004            13
           co2             gb         588              147         0.9959          0.0002            13
          temp             gb         588              147         0.9926          0.0009            13
      pressure             gb         588              147         0.9891          0.0007            13
           lux    

## 9. Metric-Level Final Scores

Combine rule-based and model-based scores per metric-window to create metric_driftscore_final and metric_faultflag_final, exporting to metric_window_scores.csv with required columns.

In [56]:
# Produces: metric_driftscore_final, metric_faultflag_final
# Model residual is used as a weak correction only — rule score dominates

# ------------------------------------------------------------------
# Note on model R²: models achieved R²>0.99 because features directly
# overlap with rule score construction. The model residual score is
# therefore used at low weight (0.15) as a soft correction only.
# The rule score (weight 0.85) is the primary signal.
# ------------------------------------------------------------------

RULE_WEIGHT  = 0.85
MODEL_WEIGHT = 0.15

df = metric_window_scores.copy()

# ------------------------------------------------------------------
# Final drift score: weighted combination
# ------------------------------------------------------------------
df['metric_driftscore_final'] = (
    RULE_WEIGHT  * df['metric_drift_score'] +
    MODEL_WEIGHT * df['metric_model_residual_score']
).clip(0, 1)

# ------------------------------------------------------------------
# Final fault flag: based on final drift score + confidence + persistence
# Same basis-specific thresholds as Cell 7
# ------------------------------------------------------------------
score_threshold = np.where(
    df['metric_basis'] == 'low_confidence', 0.85,
    np.where(df['metric_basis'] == 'self_history_only', 0.60, 0.55)
)

conf_threshold = np.where(
    df['metric_basis'] == 'self_history_only', 0.45, 0.55
)

df['metric_faultflag_final'] = (
    (df['metric_driftscore_final'] > score_threshold) &
    (df['metric_confidence'] >= conf_threshold) &
    (df['persistence_hours'] >= MIN_PERSISTENCE_HOURS)
).astype(int)

# ------------------------------------------------------------------
# Select and export required columns
# ------------------------------------------------------------------
required_cols = [
    SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL,
    'metric', 'metric_basis',
    'metric_rule_score', 'metric_model_residual_score',
    'metric_driftscore_final', 'metric_confidence',
    'metric_faultflag_final', 'neighbour_count',
    'mean_neigh_coverage', 'hours_scored',
]

metric_window_scores_final = df[required_cols].copy()
metric_window_scores_final.to_csv(DATA_DIR / 'metric_window_scores.csv', index=False)

print(f"Exported metric_window_scores.csv — {metric_window_scores_final.shape}")

# ------------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------------
print(f"\n  metric_driftscore_final stats:")
print(metric_window_scores_final['metric_driftscore_final'].describe().round(3).to_string())

print(f"\n  metric_faultflag_final rate overall: "
      f"{metric_window_scores_final['metric_faultflag_final'].mean():.3f}")

print(f"\n  Fault flag rate per metric:")
print(
    metric_window_scores_final.groupby('metric')['metric_faultflag_final']
    .mean().sort_values(ascending=False).round(3).to_string()
)

print(f"\n  Fault flag rate per metric_basis:")
print(
    metric_window_scores_final.groupby('metric_basis')['metric_faultflag_final']
    .mean().round(3).to_string()
)

print(f"\n  Rule vs final drift score correlation: "
      f"{df['metric_drift_score'].corr(df['metric_driftscore_final']):.4f}")

print(f"\n  Sample rows (tvoc):")
sample = metric_window_scores_final[
    metric_window_scores_final['metric'] == 'tvoc'
].head(4)
print(sample[[
    SENSOR_ID_COL, WINDOW_START_COL, 'metric_basis',
    'metric_rule_score', 'metric_driftscore_final',
    'metric_confidence', 'metric_faultflag_final'
]].to_string(index=False))

Exported metric_window_scores.csv — (14784, 13)

  metric_driftscore_final stats:
count    14784.000
mean         0.448
std          0.264
min          0.000
25%          0.245
50%          0.330
75%          0.691
max          1.000

  metric_faultflag_final rate overall: 0.243

  Fault flag rate per metric:
metric
lux               0.349
green_relative    0.341
snd               0.333
clear_relative    0.330
blue_relative     0.311
red_relative      0.278
humid             0.233
temp              0.209
aqi1              0.196
co2               0.196
tvoc              0.175
aqi2              0.169
voc_resistance    0.155
pressure          0.125

  Fault flag rate per metric_basis:
metric_basis
low_confidence       0.225
neighbour_aware      0.196
self_history_only    0.342

  Rule vs final drift score correlation: 0.9960

  Sample rows (tvoc):
           ateccid              window_start      metric_basis  metric_rule_score  metric_driftscore_final  metric_confidence  metric_faultflag

## 10. Metric-Level Fault Types

Assign rule-based fault type labels per metric-window (e.g., drift_gradual, stuck_sensor) based on evidence patterns like persistence and trends.

In [ ]:
# Assign rule-based fault type labels per metric-window
# Labels: normal, stuck_sensor, drift_abrupt, drift_gradual,
#         spike_transient, noisy_sensor, self_history_anomaly

# ------------------------------------------------------------------
# Re-join evidence columns needed for fault typing
# (These were dropped in Cell 9's required_cols selection)
# ------------------------------------------------------------------
evidence_cols = [
    SENSOR_ID_COL, WINDOW_START_COL, 'metric',
    'rolling_instability', 'max_self_deviation',
    'persistence_ratio', 'max_abs_residual', 'max_self_slope',
    'persistence_hours',
]

available_evidence = [c for c in evidence_cols if c in metric_window_scores.columns]

fault_df = metric_window_scores_final.merge(
    metric_window_scores[available_evidence],
    on=[SENSOR_ID_COL, WINDOW_START_COL, 'metric'],
    how='left'
)

# ------------------------------------------------------------------
# Fault type assignment function
# Priority order matters — first match wins
# ------------------------------------------------------------------
def assign_metric_fault_type(row):
    # 1. Normal — low drift score
    if row['metric_driftscore_final'] < 0.30:
        return 'normal'

    # 2. Stuck sensor — low instability + high self deviation
    # (sensor not varying but offset from own history)
    if (row.get('rolling_instability', np.nan) < INSTABILITY_THRESHOLDS['low'] and
            row.get('max_self_deviation', np.nan) > SELF_HISTORY_DEVIATION_THRESHOLD):
        return 'stuck_sensor'

    # 3. Abrupt drift — very high score + low persistence + high magnitude
    if (row['metric_driftscore_final'] > 0.80 and
            row.get('persistence_ratio', 1.0) < 0.5 and
            row.get('max_abs_residual', 0) > RESIDUAL_MAGNITUDE_THRESHOLD * 2):
        return 'drift_abrupt'

    # 4. Gradual drift — persistent + trending
    if (row.get('persistence_hours', 0) >= MIN_PERSISTENCE_HOURS and
            row.get('max_self_slope', 0) > SELF_HISTORY_SLOPE_THRESHOLD):
        return 'drift_gradual'

    # 5. Transient spike — short persistence + high magnitude
    if (row.get('persistence_hours', 0) < MIN_PERSISTENCE_HOURS and
            row.get('max_abs_residual', 0) > RESIDUAL_MAGNITUDE_THRESHOLD):
        return 'spike_transient'

    # 6. Noisy sensor — high instability
    if row.get('rolling_instability', 0) > INSTABILITY_THRESHOLDS['high']:
        return 'noisy_sensor'

    # 7. Self-history anomaly — isolated sensor with no neighbour reference
    if row['metric_basis'] == 'self_history_only':
        return 'self_history_anomaly'

    # Default
    return 'drift_gradual'


fault_df['metric_faulttype_rule'] = fault_df.apply(assign_metric_fault_type, axis=1)

# ------------------------------------------------------------------
# Merge fault type back onto metric_window_scores_final
# ------------------------------------------------------------------
metric_window_scores_final = metric_window_scores_final.merge(
    fault_df[[SENSOR_ID_COL, WINDOW_START_COL, 'metric', 'metric_faulttype_rule']],
    on=[SENSOR_ID_COL, WINDOW_START_COL, 'metric'],
    how='left'
)

print("Assigned metric-level fault types")
print(f"\n  Shape: {metric_window_scores_final.shape}")
print(f"\n  Fault type distribution:")
print(metric_window_scores_final['metric_faulttype_rule'].value_counts().to_string())

print(f"\n  Fault type by metric_basis:")
print(
    metric_window_scores_final.groupby(['metric_basis', 'metric_faulttype_rule'])
    .size().unstack(fill_value=0).to_string()
)

print(f"\n  Fault type by metric (top 5 non-normal):")
non_normal = metric_window_scores_final[
    metric_window_scores_final['metric_faulttype_rule'] != 'normal'
]
print(
    non_normal.groupby(['metric', 'metric_faulttype_rule'])
    .size().unstack(fill_value=0)
    .to_string()
)

Assigned metric-level fault types

  Shape: (14784, 14)

  Fault type distribution:
metric_faulttype_rule
normal                  6942
drift_gradual           4579
spike_transient         1946
noisy_sensor             645
drift_abrupt             582
self_history_anomaly      80
stuck_sensor              10

  Fault type by metric_basis:
metric_faulttype_rule  drift_abrupt  drift_gradual  noisy_sensor  normal  self_history_anomaly  spike_transient  stuck_sensor
metric_basis                                                                                                                 
low_confidence                  209            551            27     314                     0              235             8
neighbour_aware                 373           2215           157    4502                     0             1711             2
self_history_only                 0           1813           461    2126                    80                0             0

  Fault type by metric (top 5

## 11. Combined Sensor-Window Scoring

Aggregate metric-level outputs into final sensor-window scores, including combined_driftscore, combined_binaryfaultflag, combined_faulttype, and summary statistics like num_metrics_faulted and top_metric_score.

In [ ]:
# Aggregate metric-level outputs into final sensor-window scores
# Produces one row per sensor-window with combined drift score, fault flag,
# fault type, and summary statistics across all 14 metrics

# ------------------------------------------------------------------
# Aggregate metric scores to sensor-window level
# ------------------------------------------------------------------
sensor_window_grouped = metric_window_scores_final.groupby(
    [SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL]
)

combined_scores = sensor_window_grouped.agg(
    num_metrics_scored       = ('metric_driftscore_final', 'count'),
    top_metric_score         = ('metric_driftscore_final', 'max'),
    mean_metric_score        = ('metric_driftscore_final', 'mean'),
    median_metric_score      = ('metric_driftscore_final', 'median'),
    num_metrics_faulted      = ('metric_faultflag_final',  'sum'),
    combined_confidence      = ('metric_confidence',       'mean'),
    neighbour_count          = ('neighbour_count',         'first'),
    total_hours_scored       = ('hours_scored',            'sum'),
).reset_index()

# ------------------------------------------------------------------
# Combined drift score: mean metric score weighted by confidence
# ------------------------------------------------------------------
combined_scores['combined_drift_score'] = (
    combined_scores['mean_metric_score'] *
    combined_scores['combined_confidence']
).clip(0, 1)

# ------------------------------------------------------------------
# Combined binary fault flag
# Requires: at least 2 metrics faulted AND combined drift score above 0.45
# This raises the bar meaningfully above the mean score of 0.356
# ------------------------------------------------------------------
combined_scores['combined_binary_fault_flag'] = (
    (combined_scores['num_metrics_faulted'] >= 2) &
    (combined_scores['combined_drift_score'] > 0.45)
).astype(int)


# ------------------------------------------------------------------
# Top metric: the metric with highest drift score per sensor-window
# ------------------------------------------------------------------
top_metric_idx = (
    metric_window_scores_final
    .groupby([SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL])['metric_driftscore_final']
    .idxmax()
)

top_metrics = metric_window_scores_final.loc[
    top_metric_idx,
    [SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL,
     'metric', 'metric_driftscore_final', 'metric_faulttype_rule']
].rename(columns={
    'metric':                  'top_metric',
    'metric_driftscore_final': 'top_metric_score_check',
    'metric_faulttype_rule':   'top_metric_fault_type',
})

combined_scores = combined_scores.merge(
    top_metrics,
    on=[SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL],
    how='left'
).drop(columns=['top_metric_score_check'])

# ------------------------------------------------------------------
# Combined fault type: modal fault type across all metrics
# ------------------------------------------------------------------
fault_type_mode = (
    metric_window_scores_final
    .groupby([SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL])['metric_faulttype_rule']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={'metric_faulttype_rule': 'combined_fault_type'})
)

combined_scores = combined_scores.merge(
    fault_type_mode,
    on=[SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL],
    how='left'
)

# ------------------------------------------------------------------
# Classification basis
# ------------------------------------------------------------------
combined_scores['classification_basis'] = combined_scores['neighbour_count'].apply(
    lambda x: 'self_history_only' if x == 0
    else ('mixed' if x < 3 else 'neighbour_aware')
)

# ------------------------------------------------------------------
# Export
# ------------------------------------------------------------------
combined_scores.to_csv(DATA_DIR / 'combined_sensor_window_scores.csv', index=False)
print(f"Exported combined_sensor_window_scores.csv — {combined_scores.shape}")

# ------------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------------
print(f"\n  Expected rows: 1056 sensor-windows")
print(f"  Actual rows:   {len(combined_scores)}")

print(f"\n  combined_drift_score stats:")
print(combined_scores['combined_drift_score'].describe().round(3).to_string())

print(f"\n  combined_binary_fault_flag rate: "
      f"{combined_scores['combined_binary_fault_flag'].mean():.3f}")

print(f"\n  combined_binary_fault_flag by classification_basis:")
print(
    combined_scores.groupby('classification_basis')['combined_binary_fault_flag']
    .mean().round(3).to_string()
)

print(f"\n  combined_fault_type distribution:")
print(combined_scores['combined_fault_type'].value_counts().to_string())

print(f"\n  top_metric distribution:")
print(combined_scores['top_metric'].value_counts().to_string())

print(f"\n  num_metrics_faulted distribution:")
print(combined_scores['num_metrics_faulted'].value_counts().sort_index().to_string())

print(f"\n  Sample rows:")
print(combined_scores[[
    SENSOR_ID_COL, WINDOW_START_COL, 'classification_basis',
    'combined_drift_score', 'combined_binary_fault_flag',
    'num_metrics_faulted', 'top_metric', 'combined_fault_type'
]].head(6).to_string(index=False))

Exported combined_sensor_window_scores.csv — (1056, 17)

  Expected rows: 1056 sensor-windows
  Actual rows:   1056

  combined_drift_score stats:
count    1056.000
mean        0.356
std         0.150
min         0.036
25%         0.264
50%         0.327
75%         0.435
max         0.793

  combined_binary_fault_flag rate: 0.218

  combined_binary_fault_flag by classification_basis:
classification_basis
mixed                0.562
neighbour_aware      0.275
self_history_only    0.000

  combined_fault_type distribution:
combined_fault_type
normal             563
drift_gradual      375
spike_transient     63
drift_abrupt        30
noisy_sensor        25

  top_metric distribution:
top_metric
pressure          135
co2               125
snd               106
lux                96
tvoc               87
red_relative       79
aqi2               76
blue_relative      71
humid              68
clear_relative     52
aqi1               52
temp               47
voc_resistance     32
green_relativ

## 12. Comparison with 03c if Available

If 03c outputs exist, compare drift scores, fault flags, and fault types, identifying disagreements and exporting a comparison table to combined_vs_03c_comparison.csv.

In [ ]:
# Compare Cell 11 combined scores against 03c outputs if available
# Identifies agreements, disagreements, and drift score differences

if sensor_window_scores_03c is not None:
    print("=== Comparing with 03c scores ===")

    # Normalise join keys
    combined_scores[WINDOW_START_COL] = pd.to_datetime(
        combined_scores[WINDOW_START_COL], utc=True
    )
    sensor_window_scores_03c[WINDOW_START_COL] = pd.to_datetime(
        sensor_window_scores_03c[WINDOW_START_COL], utc=True
    )

    # Identify available 03c columns
    drift_col_03c = next(
        (c for c in sensor_window_scores_03c.columns
         if 'drift' in c.lower() and 'score' in c.lower()), None
    )
    flag_col_03c = next(
        (c for c in sensor_window_scores_03c.columns
         if 'fault' in c.lower() and 'flag' in c.lower()), None
    )
    type_col_03c = next(
        (c for c in sensor_window_scores_03c.columns
         if 'fault' in c.lower() and 'type' in c.lower()), None
    )

    print(f"  03c drift col:  {drift_col_03c}")
    print(f"  03c flag col:   {flag_col_03c}")
    print(f"  03c type col:   {type_col_03c}")

    # Select 03c cols to merge
    merge_cols_03c = [SENSOR_ID_COL, WINDOW_START_COL]
    rename_03c = {}
    for col, label in [(drift_col_03c, 'combined_drift_score'),
                       (flag_col_03c, 'combined_binary_fault_flag'),
                       (type_col_03c, 'combined_fault_type')]:
        if col:
            merge_cols_03c.append(col)
            rename_03c[col] = f"{label}_03c"

    # Rename 03c sensor ID column to match our convention
    sensor_window_scores_03c = sensor_window_scores_03c.rename(
        columns={'sensor_id': SENSOR_ID_COL}
    )

    comparison = combined_scores.merge(
        sensor_window_scores_03c[merge_cols_03c].rename(columns=rename_03c),
        on=[SENSOR_ID_COL, WINDOW_START_COL],
        suffixes=('_04', ''),
        how='inner'
    )

    print(f"\n  Total windows compared: {len(comparison)}")

    # Drift score difference
    if 'combined_drift_score_03c' in comparison.columns:
        comparison['drift_score_diff'] = (
            comparison['combined_drift_score'] - comparison['combined_drift_score_03c']
        )
        print(f"\n  Drift score difference (04 - 03c):")
        print(comparison['drift_score_diff'].describe().round(3).to_string())

    # Fault flag agreement
    if 'combined_binary_fault_flag_03c' in comparison.columns:
        comparison['flag_agreement'] = (
            comparison['combined_binary_fault_flag'] ==
            comparison['combined_binary_fault_flag_03c']
        ).astype(int)
        print(f"\n  Fault flag agreement rate: {comparison['flag_agreement'].mean():.3f}")

        # Confusion breakdown
        tp = ((comparison['combined_binary_fault_flag'] == 1) &
              (comparison['combined_binary_fault_flag_03c'] == 1)).sum()
        tn = ((comparison['combined_binary_fault_flag'] == 0) &
              (comparison['combined_binary_fault_flag_03c'] == 0)).sum()
        fp = ((comparison['combined_binary_fault_flag'] == 1) &
              (comparison['combined_binary_fault_flag_03c'] == 0)).sum()
        fn = ((comparison['combined_binary_fault_flag'] == 0) &
              (comparison['combined_binary_fault_flag_03c'] == 1)).sum()
        print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")

    # Major disagreements
    major_disagreements = comparison[
        (comparison.get('drift_score_diff', pd.Series(dtype=float)).abs() > 0.3) |
        (comparison.get('flag_agreement', pd.Series(dtype=int)) == 0)
    ] if 'drift_score_diff' in comparison.columns else comparison[
        comparison.get('flag_agreement', pd.Series(dtype=int)) == 0
    ]
    print(f"\n  Major disagreements: {len(major_disagreements)}")

    # Export comparison
    comparison.to_csv(DATA_DIR / 'combined_vs_03c_comparison.csv', index=False)
    print(f"  Exported combined_vs_03c_comparison.csv — {comparison.shape}")

else:
    print("03c outputs not available — skipping comparison")
    print("Proceeding to Cell 13 (Visual Diagnostics)")

    # Export combined scores only
    combined_scores.to_csv(DATA_DIR / 'combined_sensor_window_scores.csv', index=False)
    print(f"combined_sensor_window_scores.csv already exported — {combined_scores.shape}")

=== Comparing with 03c scores ===
  03c drift col:  drift_score
  03c flag col:   binary_fault_flag
  03c type col:   None

  Total windows compared: 1056

  Drift score difference (04 - 03c):
count    1056.000
mean       -0.451
std         0.178
min        -0.864
25%        -0.608
50%        -0.489
75%        -0.299
max        -0.075

  Fault flag agreement rate: 0.524
  TP=230  TN=323  FP=0  FN=503

  Major disagreements: 790
  Exported combined_vs_03c_comparison.csv — (1056, 21)


## 13. Visual Diagnostics

Generate plots including per-metric drift distributions, heatmaps, time-series examples, comparison charts with 03c, and stacked fault type bars, saving to /data/ or /output/.

In [65]:
# Visual diagnostics
# Plots saved to OUTPUT_DIR:
#   1. Per-metric drift score distributions
#   2. Sensor-window combined drift score heatmap
#   3. Fault type stacked bar by metric
#   4. 04 vs 03c drift score comparison scatter
#   5. Time-series example: top drifting sensor

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=0.9)
PLOT_COLS = 3
FIG_DPI = 150

# ------------------------------------------------------------------
# 1. Per-metric drift score distributions (box plots)
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))

metric_order = (
    metric_window_scores_final
    .groupby('metric')['metric_driftscore_final']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

plot_data = [
    metric_window_scores_final.loc[
        metric_window_scores_final['metric'] == m, 'metric_driftscore_final'
    ].values
    for m in metric_order
]

bp = ax.boxplot(plot_data, labels=metric_order, patch_artist=True,
                medianprops=dict(color='#01696f', linewidth=2))
for patch in bp['boxes']:
    patch.set_facecolor('#cedcd8')
    patch.set_alpha(0.7)

ax.axhline(0.55, color='#a12c7b', linestyle='--', linewidth=1, label='Score threshold (0.55)')
ax.set_xlabel('Metric')
ax.set_ylabel('Metric Drift Score (final)')
ax.set_title('Per-Metric Drift Score Distributions')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_metric_drift_distributions.png', dpi=FIG_DPI)
plt.close()
print("Saved: 04_metric_drift_distributions.png")

# ------------------------------------------------------------------
# 2. Sensor-window heatmap: combined drift score over time
# ------------------------------------------------------------------
pivot = combined_scores.pivot_table(
    index=SENSOR_ID_COL,
    columns=WINDOW_START_COL,
    values='combined_drift_score'
)

# Trim sensor ID to last 8 chars for readability
pivot.index = [str(i)[-8:] for i in pivot.index]
# Show every 7th window label
n_windows = pivot.shape[1]
xtick_step = max(1, n_windows // 8)

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(
    pivot, ax=ax,
    cmap='YlOrRd', vmin=0, vmax=0.8,
    xticklabels=xtick_step,
    yticklabels=True,
    linewidths=0,
    cbar_kws={'label': 'Combined Drift Score'}
)
ax.set_title('Sensor-Window Combined Drift Score Heatmap')
ax.set_xlabel('Window Start')
ax.set_ylabel('Sensor (last 8 chars)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_drift_score_heatmap.png', dpi=FIG_DPI)
plt.close()
print("Saved: 04_drift_score_heatmap.png")

# ------------------------------------------------------------------
# 3. Fault type stacked bar by metric
# ------------------------------------------------------------------
fault_type_counts = (
    metric_window_scores_final
    .groupby(['metric', 'metric_faulttype_rule'])
    .size()
    .unstack(fill_value=0)
)
fault_type_counts = fault_type_counts.loc[metric_order]

fault_colors = {
    'normal':               '#d4d1ca',
    'drift_gradual':        '#01696f',
    'drift_abrupt':         '#da7101',
    'spike_transient':      '#d19900',
    'noisy_sensor':         '#a12c7b',
    'stuck_sensor':         '#006494',
    'self_history_anomaly': '#964219',
}

fig, ax = plt.subplots(figsize=(14, 6))
bottom = None
for ftype in ['normal', 'drift_gradual', 'drift_abrupt',
              'spike_transient', 'noisy_sensor', 'stuck_sensor',
              'self_history_anomaly']:
    if ftype not in fault_type_counts.columns:
        continue
    vals = fault_type_counts[ftype].values
    ax.bar(
        fault_type_counts.index, vals,
        bottom=bottom if bottom is not None else 0,
        label=ftype, color=fault_colors.get(ftype, '#999'),
        width=0.7
    )
    bottom = vals if bottom is None else bottom + vals

ax.set_xlabel('Metric')
ax.set_ylabel('Window Count')
ax.set_title('Fault Type Distribution by Metric')
ax.tick_params(axis='x', rotation=45)
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_fault_type_by_metric.png', dpi=FIG_DPI)
plt.close()
print("Saved: 04_fault_type_by_metric.png")

# ------------------------------------------------------------------
# 4. 04 vs 03c drift score scatter (if comparison available)
# ------------------------------------------------------------------
try:
    comp_df = pd.read_csv(DATA_DIR / 'combined_vs_03c_comparison.csv')
    if 'combined_drift_score_03c' in comp_df.columns:
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.scatter(
            comp_df['combined_drift_score_03c'],
            comp_df['combined_drift_score'],
            alpha=0.3, s=15, color='#01696f'
        )
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect agreement')
        ax.set_xlabel('03c Drift Score')
        ax.set_ylabel('04 Combined Drift Score')
        ax.set_title('04 vs 03c Drift Score Comparison')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / '04_vs_03c_scatter.png', dpi=FIG_DPI)
        plt.close()
        print("Saved: 04_vs_03c_scatter.png")
except Exception as e:
    print(f"Skipped 03c scatter: {e}")

# ------------------------------------------------------------------
# 5. Time-series: top drifting sensor across all metrics
# ------------------------------------------------------------------
top_sensor = (
    combined_scores
    .groupby(SENSOR_ID_COL)['combined_drift_score']
    .mean()
    .idxmax()
)

sensor_ts = metric_window_scores_final[
    metric_window_scores_final[SENSOR_ID_COL] == top_sensor
].copy()
sensor_ts[WINDOW_START_COL] = pd.to_datetime(sensor_ts[WINDOW_START_COL], utc=True)

fig, axes = plt.subplots(
    len(metrics), 1,
    figsize=(16, len(metrics) * 1.5),
    sharex=True
)

for i, metric in enumerate(metric_order):
    ax = axes[i]
    mdf = sensor_ts[sensor_ts['metric'] == metric].sort_values(WINDOW_START_COL)
    ax.plot(
        mdf[WINDOW_START_COL], mdf['metric_driftscore_final'],
        linewidth=1.2, color='#01696f'
    )
    # Shade flagged windows
    flagged = mdf[mdf['metric_faultflag_final'] == 1]
    for _, row in flagged.iterrows():
        ax.axvspan(row[WINDOW_START_COL],
                   row[WINDOW_START_COL] + pd.Timedelta(hours=24),
                   alpha=0.2, color='#da7101')
    ax.set_ylabel(metric, fontsize=7, rotation=0, labelpad=60, va='center')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='y', labelsize=7)
    ax.axhline(0.55, color='#a12c7b', linestyle=':', linewidth=0.8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].tick_params(axis='x', rotation=30, labelsize=8)
fig.suptitle(
    f'Per-Metric Drift Score Time Series — {str(top_sensor)[-12:]}',
    fontsize=11, y=1.001
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_top_sensor_timeseries.png', dpi=FIG_DPI, bbox_inches='tight')
plt.close()
print(f"Saved: 04_top_sensor_timeseries.png  (sensor: {top_sensor})")

print("\nAll diagnostic plots saved.")

Saved: 04_metric_drift_distributions.png
Saved: 04_drift_score_heatmap.png
Saved: 04_fault_type_by_metric.png
Saved: 04_vs_03c_scatter.png
Saved: 04_top_sensor_timeseries.png  (sensor: 012330E186CB5BA801)

All diagnostic plots saved.


## 14. Export Outputs

Write final outputs to sensor_window_metric_first_scores.csv, metric_window_scores.csv, metric_window_evidence.csv, metric_model_summary.csv, metric_registry.csv, and top_metric_first_fault_windows.csv.

In [ ]:
# Main combined output
combined_output_cols = [
    'ateccid', 'window_start', 'window_end', 'combined_driftscore', 'combined_binaryfaultflag',
    'combined_faulttype', 'combined_confidence', 'num_metrics_scored', 'num_metrics_faulted',
    'top_metric', 'top_metric_score', 'top_metric_faulttype', 'mean_metric_score',
    'median_metric_score', 'max_metric_score', 'classification_basis'
]
sensor_window_metric_first_scores = combined_scores[combined_output_cols]
sensor_window_metric_first_scores.to_csv(DATA_DIR / 'sensor_window_metric_first_scores.csv', index=False)

# Top fault windows
top_fault_windows = combined_scores.nlargest(100, 'combined_driftscore')[
    ['ateccid', 'window_start', 'window_end', 'combined_driftscore', 'top_metric', 'top_metric_score', 'num_metrics_faulted']
]
top_fault_windows.to_csv(DATA_DIR / 'top_metric_first_fault_windows.csv', index=False)

print("Exported all final outputs:")
print("- sensor_window_metric_first_scores.csv")
print("- metric_window_scores.csv (already exported)")
print("- metric_window_evidence.csv (already exported)")
print("- metric_model_summary.csv (already exported)")
print("- metric_registry.csv (already exported)")
print("- top_metric_first_fault_windows.csv")

In [66]:
# Final export of all required outputs for notebook 04
# Produces clean, well-labelled CSV files and a summary JSON

# ------------------------------------------------------------------
# 1. metric_window_scores.csv — already exported in Cell 9
#    Re-export with fault type added from Cell 10
# ------------------------------------------------------------------
final_metric_cols = [
    SENSOR_ID_COL, WINDOW_START_COL, WINDOW_END_COL,
    'metric', 'metric_basis',
    'metric_rule_score', 'metric_model_residual_score',
    'metric_driftscore_final', 'metric_confidence',
    'metric_faultflag_final', 'metric_faulttype_rule',
    'neighbour_count', 'mean_neigh_coverage', 'hours_scored',
]

metric_window_scores_final[final_metric_cols].to_csv(
    DATA_DIR / 'metric_window_scores.csv', index=False
)
print(f"Exported metric_window_scores.csv — "
      f"{metric_window_scores_final[final_metric_cols].shape}")

# ------------------------------------------------------------------
# 2. combined_sensor_window_scores.csv — re-export clean version
# ------------------------------------------------------------------
combined_scores.to_csv(DATA_DIR / 'combined_sensor_window_scores.csv', index=False)
print(f"Exported combined_sensor_window_scores.csv — {combined_scores.shape}")

# ------------------------------------------------------------------
# 3. metric_model_summary.csv — already exported in Cell 8
# ------------------------------------------------------------------
print(f"metric_model_summary.csv already exported — "
      f"{metric_model_summary_df.shape}")

# ------------------------------------------------------------------
# 4. metric_registry.csv — already exported in Cell 4
# ------------------------------------------------------------------
print(f"metric_registry.csv already exported — "
      f"{pd.read_csv(DATA_DIR / 'metric_registry.csv').shape}")

# ------------------------------------------------------------------
# 5. fault_rule_parameters.json — document all thresholds used
# ------------------------------------------------------------------
fault_rule_parameters = {
    'notebook':                      '04_metric_first_fault_modelling',
    'robust_z_threshold':             ROBUST_Z_THRESHOLD,
    'self_history_z_threshold':       SELF_HISTORY_Z_THRESHOLD,
    'residual_magnitude_threshold':   RESIDUAL_MAGNITUDE_THRESHOLD,
    'self_history_deviation_threshold': SELF_HISTORY_DEVIATION_THRESHOLD,
    'self_history_slope_threshold':   SELF_HISTORY_SLOPE_THRESHOLD,
    'persistence_ratio_threshold':    PERSISTENCE_RATIO_THRESHOLD,
    'min_persistence_hours':          MIN_PERSISTENCE_HOURS,
    'persistence_window_hours':       PERSISTENCE_WINDOW_HOURS,
    'min_neighbour_coverage':         MIN_NEIGHBOUR_COVERAGE,
    'isolated_sensor_discount':       ISOLATED_SENSOR_DISCOUNT,
    'instability_thresholds':         INSTABILITY_THRESHOLDS,
    'decision_window_hours':          DECISION_WINDOW_HOURS,
    'rule_weight':                    RULE_WEIGHT,
    'model_weight':                   MODEL_WEIGHT,
    'fault_flag_score_threshold': {
        'neighbour_aware':   0.55,
        'low_confidence':    0.85,
        'self_history_only': 0.60,
    },
    'fault_flag_confidence_threshold': {
        'neighbour_aware':   0.55,
        'low_confidence':    0.55,
        'self_history_only': 0.45,
    },
    'combined_fault_flag': {
        'min_metrics_faulted': 2,
        'min_combined_drift_score': 0.45,
    }
}

with open(DATA_DIR / 'fault_rule_parameters.json', 'w') as f:
    json.dump(fault_rule_parameters, f, indent=2)
print(f"Exported fault_rule_parameters.json")

# ------------------------------------------------------------------
# 6. score_diagnostics.csv — per-metric summary statistics
# ------------------------------------------------------------------
score_diagnostics = []

for metric in metrics:
    mdf = metric_window_scores_final[metric_window_scores_final['metric'] == metric]
    score_diagnostics.append({
        'metric':                  metric,
        'mean_drift_score':        round(mdf['metric_driftscore_final'].mean(), 4),
        'median_drift_score':      round(mdf['metric_driftscore_final'].median(), 4),
        'p90_drift_score':         round(mdf['metric_driftscore_final'].quantile(0.90), 4),
        'fault_flag_rate':         round(mdf['metric_faultflag_final'].mean(), 4),
        'mean_confidence':         round(mdf['metric_confidence'].mean(), 4),
        'dominant_fault_type':     mdf['metric_faulttype_rule'].mode().iloc[0],
        'pct_normal':              round((mdf['metric_faulttype_rule'] == 'normal').mean(), 4),
        'pct_drift_gradual':       round((mdf['metric_faulttype_rule'] == 'drift_gradual').mean(), 4),
        'pct_spike_transient':     round((mdf['metric_faulttype_rule'] == 'spike_transient').mean(), 4),
        'pct_noisy_sensor':        round((mdf['metric_faulttype_rule'] == 'noisy_sensor').mean(), 4),
    })

score_diagnostics_df = pd.DataFrame(score_diagnostics)
score_diagnostics_df.to_csv(DATA_DIR / 'score_diagnostics.csv', index=False)
print(f"Exported score_diagnostics.csv — {score_diagnostics_df.shape}")
print(score_diagnostics_df.to_string(index=False))

# ------------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------------
print(f"""
=== Notebook 04 Export Summary ===
  metric_window_scores.csv         {len(metric_window_scores_final):>6} rows  (14 metrics × 1056 windows)
  combined_sensor_window_scores.csv {len(combined_scores):>5} rows  (1 row per sensor-window)
  metric_model_summary.csv         {len(metric_model_summary_df):>6} rows  (1 row per metric)
  metric_registry.csv              {len(metric_registry):>6} rows  (1 row per metric)
  fault_rule_parameters.json              (threshold documentation)
  score_diagnostics.csv            {len(score_diagnostics_df):>6} rows  (per-metric diagnostics)

  Overall combined fault flag rate:  {combined_scores['combined_binary_fault_flag'].mean():.3f}
  Overall metric fault flag rate:    {metric_window_scores_final['metric_faultflag_final'].mean():.3f}
  Sensors processed:                 {combined_scores[SENSOR_ID_COL].nunique()}
  Windows per sensor:                {combined_scores.groupby(SENSOR_ID_COL).size().iloc[0]}
  Date range:                        {combined_scores[WINDOW_START_COL].min()} → {combined_scores[WINDOW_START_COL].max()}
""")

Exported metric_window_scores.csv — (14784, 14)
Exported combined_sensor_window_scores.csv — (1056, 17)
metric_model_summary.csv already exported — (14, 7)
metric_registry.csv already exported — (14, 27)
Exported fault_rule_parameters.json
Exported score_diagnostics.csv — (14, 11)
        metric  mean_drift_score  median_drift_score  p90_drift_score  fault_flag_rate  mean_confidence dominant_fault_type  pct_normal  pct_drift_gradual  pct_spike_transient  pct_noisy_sensor
          tvoc            0.4058              0.3487           0.7997           0.1752           0.8084              normal      0.4659             0.3305               0.0634            0.0748
           co2            0.4169              0.2779           0.8255           0.1960           0.8084              normal      0.5275             0.2614               0.1449            0.0445
          temp            0.3773              0.2416           0.8546           0.2093           0.8084              normal      0.5928 